# Predictive Modeling

The purpose of this notebook is to develope machine learning models capable of predicting CMS Hospital Overall Ratings using hospital characteristics, patient experience measures, infection metrics, mortality measures, and readmission measures derived during exploratory data analysis and feature engineering.

Three supervised machine learning algorithms will be developed and compared:
    -Logistic Regression
    -Decision Tree
    -Random Forest

The resulting models will be evaluated in 05_model_evaluation.ipynb.

# Imports

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# Load the Master Dataset

In [2]:
ROOT_DIR = Path.cwd().parent

PROCESSED_DIR = ROOT_DIR / "data" / "processed"

master_df = pd.read_csv(
    PROCESSED_DIR / "master_dataset.csv"
)

print(master_df.shape)
display(master_df.head())

(5432, 44)


,facility_id,facility_name,address,city_town,state,zip_code,county_parish,telephone_number,hospital_type,hospital_ownership,...,pt_exp_group_footnote,te_group_measure_count,count_of_facility_te_measures,te_group_footnote,heart_attack_readmit,cabg_readmit,copd_readmit,heart_failure_readmit,hip_knee_readmit,pneumonia_readmit
0,10001,SOUTHEAST HEALTH MEDICAL CENTER,1108 ROSS CLARK CIRCLE,DOTHAN,AL,36301,HOUSTON,(334) 793-8701,Acute Care Hospitals,Government - Hospital District or Authority,...,NaN,10.0,10.0,NaN,13.0,10.1,18.0,20.1,4.8,16.0
1,10005,MARSHALL MEDICAL CENTERS,2505 U S HIGHWAY 431 NORTH,BOAZ,AL,35957,MARSHALL,(256) 593-8310,Acute Care Hospitals,Government - Hospital District or Authority,...,NaN,10.0,10.0,NaN,NaN,NaN,17.1,19.8,4.2,13.9
2,10006,NORTH ALABAMA MEDICAL CENTER,1701 VETERANS DRIVE,FLORENCE,AL,35630,LAUDERDALE,(256) 768-8400,Acute Care Hospitals,Proprietary,...,NaN,10.0,9.0,NaN,12.5,10.6,19.1,19.5,5.1,15.7
3,10007,MIZELL MEMORIAL HOSPITAL,702 N MAIN ST,OPP,AL,36467,COVINGTON,(334) 493-3541,Acute Care Hospitals,Voluntary non-profit - Private,...,NaN,10.0,7.0,NaN,NaN,NaN,18.6,20.9,NaN,16.5
4,10011,ST. VINCENT'S EAST,50 MEDICAL PARK EAST DRIVE,BIRMINGHAM,AL,35235,JEFFERSON,(205) 838-3122,Acute Care Hospitals,Voluntary non-profit - Private,...,29.0,10.0,7.0,29.0,13.2,11.9,18.2,20.9,NaN,16.2


# Confirm values of Master Dataset

In [3]:
master_df["hospital_overall_rating"].value_counts(dropna=False).sort_index()

hospital_overall_rating
1.0     199
2.0     662
3.0     987
4.0     950
5.0     384
NaN    2250
Name: count, dtype: int64

# Remove rows with missing target value (Overall Rating/NaN)
Removing rows with NaN ensures better results by only focusing on rows containing the target value. 

In [4]:
model_df = master_df.dropna(subset=["hospital_overall_rating"]).copy()

print(model_df.shape)

(3182, 44)


# Identify features

In [5]:
model_df.info()

<class 'pandas.DataFrame'>
Index: 3182 entries, 0 to 5408
Data columns (total 44 columns):
 #   Column                                            Non-Null Count  Dtype  
---  ------                                            --------------  -----  
 0   facility_id                                       3182 non-null   str    
 1   facility_name                                     3182 non-null   str    
 2   address                                           3182 non-null   str    
 3   city_town                                         3182 non-null   str    
 4   state                                             3182 non-null   str    
 5   zip_code                                          3182 non-null   int64  
 6   county_parish                                     3182 non-null   str    
 7   telephone_number                                  3182 non-null   str    
 8   hospital_type                                     3182 non-null   str    
 9   hospital_ownership                 

In [6]:
model_df.head()

,facility_id,facility_name,address,city_town,state,zip_code,county_parish,telephone_number,hospital_type,hospital_ownership,...,pt_exp_group_footnote,te_group_measure_count,count_of_facility_te_measures,te_group_footnote,heart_attack_readmit,cabg_readmit,copd_readmit,heart_failure_readmit,hip_knee_readmit,pneumonia_readmit
0,10001,SOUTHEAST HEALTH MEDICAL CENTER,1108 ROSS CLARK CIRCLE,DOTHAN,AL,36301,HOUSTON,(334) 793-8701,Acute Care Hospitals,Government - Hospital District or Authority,...,NaN,10.0,10.0,NaN,13.0,10.1,18.0,20.1,4.8,16.0
1,10005,MARSHALL MEDICAL CENTERS,2505 U S HIGHWAY 431 NORTH,BOAZ,AL,35957,MARSHALL,(256) 593-8310,Acute Care Hospitals,Government - Hospital District or Authority,...,NaN,10.0,10.0,NaN,NaN,NaN,17.1,19.8,4.2,13.9
2,10006,NORTH ALABAMA MEDICAL CENTER,1701 VETERANS DRIVE,FLORENCE,AL,35630,LAUDERDALE,(256) 768-8400,Acute Care Hospitals,Proprietary,...,NaN,10.0,9.0,NaN,12.5,10.6,19.1,19.5,5.1,15.7
3,10007,MIZELL MEMORIAL HOSPITAL,702 N MAIN ST,OPP,AL,36467,COVINGTON,(334) 493-3541,Acute Care Hospitals,Voluntary non-profit - Private,...,NaN,10.0,7.0,NaN,NaN,NaN,18.6,20.9,NaN,16.5
4,10011,ST. VINCENT'S EAST,50 MEDICAL PARK EAST DRIVE,BIRMINGHAM,AL,35235,JEFFERSON,(205) 838-3122,Acute Care Hospitals,Voluntary non-profit - Private,...,29.0,10.0,7.0,29.0,13.2,11.9,18.2,20.9,NaN,16.2


# Convert float values to integers
This will allow scikit-learn to treat them as labels.

In [7]:
model_df["hospital_overall_rating"] = (
    model_df["hospital_overall_rating"].astype(int)
)

model_df["hospital_overall_rating"].value_counts().sort_index()

hospital_overall_rating
1    199
2    662
3    987
4    950
5    384
Name: count, dtype: int64

# Remove unnecessary columns
Remove columns that should not be included as predictors.

In [9]:
columns_to_drop = [
    "facility_id",
    "facility_name",
    "address",
    "city_town",
    "telephone_number",
    "zip_code",
    "hospital_overall_rating",          # target (remove from X only)
    "hospital_overall_rating_footnote",
    "mort_group_footnote",
]

In [10]:
print(model_df.columns.tolist())

['facility_id', 'facility_name', 'address', 'city_town', 'state', 'zip_code', 'county_parish', 'telephone_number', 'hospital_type', 'hospital_ownership', 'emergency_services', 'meets_criteria_for_birthing_friendly_designation', 'hospital_overall_rating', 'hospital_overall_rating_footnote', 'mort_group_measure_count', 'count_of_facility_mort_measures', 'count_of_mort_measures_better', 'count_of_mort_measures_no_different', 'count_of_mort_measures_worse', 'mort_group_footnote', 'safety_group_measure_count', 'count_of_facility_safety_measures', 'count_of_safety_measures_better', 'count_of_safety_measures_no_different', 'count_of_safety_measures_worse', 'safety_group_footnote', 'readm_group_measure_count', 'count_of_facility_readm_measures', 'count_of_readm_measures_better', 'count_of_readm_measures_no_different', 'count_of_readm_measures_worse', 'readm_group_footnote', 'pt_exp_group_measure_count', 'count_of_facility_pt_exp_measures', 'pt_exp_group_footnote', 'te_group_measure_count', 'co

In [11]:
target = "hospital_overall_rating"

In [12]:
categorical_features = [
    "city_town",
    "state",
    "county_parish",
    "hospital_type",
    "hospital_ownership",
    "emergency_services",
    "meets_criteria_for_birthing_friendly_designation",
]

In [13]:
summary_columns = [
    "mort_group_measure_count",
    "count_of_facility_mort_measures",
    "count_of_mort_measures_better",
    "count_of_mort_measures_no_different",
    "count_of_mort_measures_worse",

    "safety_group_measure_count",
    "count_of_facility_safety_measures",
    "count_of_safety_measures_better",
    "count_of_safety_measures_no_different",
    "count_of_safety_measures_worse",

    "readm_group_measure_count",
    "count_of_facility_readm_measures",
    "count_of_readm_measures_better",
    "count_of_readm_measures_no_different",
    "count_of_readm_measures_worse",

    "pt_exp_group_measure_count",
    "count_of_facility_pt_exp_measures",

    "te_group_measure_count",
    "count_of_facility_te_measures",

    "mort_group_footnote",
    "safety_group_footnote",
    "readm_group_footnote",
    "pt_exp_group_footnote",
    "te_group_footnote",
]

In [14]:
numeric_features = [
    "heart_attack_readmit",
    "cabg_readmit",
    "copd_readmit",
    "heart_failure_readmit",
    "hip_knee_readmit",
    "pneumonia_readmit",
]

# Define columns to drop and define Summary Columns

In [16]:
drop_columns = [
    "facility_id",
    "facility_name",
    "address",
    "telephone_number",
    "hospital_overall_rating",
    "hospital_overall_rating_footnote",
    "zip_code",
]

In [17]:
summary_columns = [
    "mort_group_measure_count",
    "count_of_facility_mort_measures",
    "count_of_mort_measures_better",
    "count_of_mort_measures_no_different",
    "count_of_mort_measures_worse",

    "safety_group_measure_count",
    "count_of_facility_safety_measures",
    "count_of_safety_measures_better",
    "count_of_safety_measures_no_different",
    "count_of_safety_measures_worse",

    "readm_group_measure_count",
    "count_of_facility_readm_measures",
    "count_of_readm_measures_better",
    "count_of_readm_measures_no_different",
    "count_of_readm_measures_worse",

    "pt_exp_group_measure_count",
    "count_of_facility_pt_exp_measures",

    "te_group_measure_count",
    "count_of_facility_te_measures",

    "mort_group_footnote",
    "safety_group_footnote",
    "readm_group_footnote",
    "pt_exp_group_footnote",
    "te_group_footnote",
]

# Build X & Y 
I've cleaned the columns and identified my target and categories. Next step builds the x & y models.

In [18]:
target = "hospital_overall_rating"

exclude = drop_columns + summary_columns

X = model_df.drop(columns=exclude)
y = model_df[target].astype(int)

print("X shape:", X.shape)
print("y shape:", y.shape)

display(X.head())

X shape: (3182, 13)
y shape: (3182,)


,city_town,state,county_parish,hospital_type,hospital_ownership,emergency_services,meets_criteria_for_birthing_friendly_designation,heart_attack_readmit,cabg_readmit,copd_readmit,heart_failure_readmit,hip_knee_readmit,pneumonia_readmit
0,DOTHAN,AL,HOUSTON,Acute Care Hospitals,Government - Hospital District or Authority,Yes,Y,13.0,10.1,18.0,20.1,4.8,16.0
1,BOAZ,AL,MARSHALL,Acute Care Hospitals,Government - Hospital District or Authority,Yes,Y,NaN,NaN,17.1,19.8,4.2,13.9
2,FLORENCE,AL,LAUDERDALE,Acute Care Hospitals,Proprietary,Yes,Y,12.5,10.6,19.1,19.5,5.1,15.7
3,OPP,AL,COVINGTON,Acute Care Hospitals,Voluntary non-profit - Private,Yes,NaN,NaN,NaN,18.6,20.9,NaN,16.5
4,BIRMINGHAM,AL,JEFFERSON,Acute Care Hospitals,Voluntary non-profit - Private,Yes,NaN,13.2,11.9,18.2,20.9,NaN,16.2


In [19]:
[col for col in model_df.columns if "star" in col.lower()]

[]